# Endurance Online — Scraper Completo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pier-car/endurance-scraper/blob/main/endurance_scraper.ipynb)

Scraper completo del sito **enduranceonline.it** (sistema PHP legacy senza API).
Estrae anagrafica cavalli, risultati storici delle gare (dal 2004), classifiche mondiali
e indice gare, salvando tutto in CSV (UTF-8 BOM).

**Endpoint coperti:**
- `/db/db_horses.php` — Anagrafica cavalli (`?ricerca_n=LETTERA`)
- `/db/db_results.php` — Risultati storici gare
- `/db/db_champions.php` — Rankings mondiali (`?v_filtro=...`)
- `/db/db.php` — Database generale rider/horse
- `/live/live.php` — Live tracking (snapshot)

**Output CSV:**
- `horses_registry.csv` — anagrafica cavalli + genealogia
- `results_full.csv` — tutti i risultati storici
- `rankings.csv` — classifiche e ranking mondiali
- `races_index.csv` — indice di tutte le gare

Imposta `TEST_MODE = True` nella cella di configurazione per un giro veloce di prova.

## 1. Installazione dipendenze

In [ ]:
# Installazione dipendenze (idempotente, funziona su Colab e in locale)
import sys, subprocess
def _pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import requests, bs4, pandas, lxml, tqdm  # noqa: F401
except ImportError:
    _pip_install(["requests>=2.31.0", "beautifulsoup4>=4.12.0",
                  "pandas>=2.0.0", "lxml>=4.9.0", "tqdm>=4.66.0"])
print("Dipendenze pronte.")

## 2. Configurazione globale

Modifica `TEST_MODE` per cambiare il volume di scraping.

In [ ]:
import os, re, time, random, string, datetime as dt
from urllib.parse import urljoin, urlencode, urlparse, parse_qs
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm.auto import tqdm

# ----------------------- CONFIG -----------------------
TEST_MODE   = True                # True = giro rapido (poche lettere/anni); False = full
BASE_URL    = "https://www.enduranceonline.it"
OUT_DIR     = Path("./data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

REQUEST_DELAY   = 0.8             # secondi tra richieste (politeness)
REQUEST_TIMEOUT = 30
MAX_RETRIES     = 4
START_YEAR      = 2004
END_YEAR        = dt.date.today().year

# In modalità test riduciamo il volume
if TEST_MODE:
    LETTERS_TO_SCRAPE  = ["A", "B"]
    YEARS_TO_SCRAPE    = list(range(END_YEAR - 1, END_YEAR + 1))
    MAX_PAGES_PER_LIST = 2
    MAX_HORSE_DETAILS  = 25
else:
    LETTERS_TO_SCRAPE  = list(string.ascii_uppercase)
    YEARS_TO_SCRAPE    = list(range(START_YEAR, END_YEAR + 1))
    MAX_PAGES_PER_LIST = 200
    MAX_HORSE_DETAILS  = None     # nessun limite

# Filtri ranking noti / da testare
RANKING_FILTERS = [
    "Open Riders World Ranking",
    "Open Horses World Ranking",
    "Young Riders World Ranking",
    "Junior Riders World Ranking",
    "Italian Riders Ranking",
    "Italian Horses Ranking",
]

USER_AGENT = ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")

session = requests.Session()
session.headers.update({
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "it-IT,it;q=0.9,en;q=0.8",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
})

print(f"TEST_MODE={TEST_MODE} | anni={YEARS_TO_SCRAPE[0]}..{YEARS_TO_SCRAPE[-1]} "
      f"| lettere={len(LETTERS_TO_SCRAPE)} | output={OUT_DIR.resolve()}")

## 3. Helper HTTP (retry, delay, parsing)

In [ ]:
def fetch(url, params=None, method="GET", data=None, _attempt=1):
    """GET/POST con retry esponenziale e delay polite."""
    try:
        if method.upper() == "POST":
            r = session.post(url, data=data, timeout=REQUEST_TIMEOUT)
        else:
            r = session.get(url, params=params, timeout=REQUEST_TIMEOUT)
        # Forza UTF-8: il sito serve Latin-1 in alcuni casi ma il contenuto è UTF-8.
        if not r.encoding or r.encoding.lower() in ("iso-8859-1", "latin-1"):
            r.encoding = r.apparent_encoding or "utf-8"
        if r.status_code >= 500:
            raise requests.HTTPError(f"HTTP {r.status_code}")
        r.raise_for_status()
        time.sleep(REQUEST_DELAY + random.uniform(0, 0.2))
        return r
    except (requests.RequestException, requests.HTTPError) as e:
        if _attempt >= MAX_RETRIES:
            print(f"  ! fetch fallita {url} params={params}: {e}")
            return None
        backoff = (2 ** _attempt) + random.uniform(0, 1)
        time.sleep(backoff)
        return fetch(url, params=params, method=method, data=data, _attempt=_attempt + 1)

def soup_of(resp):
    if resp is None:
        return None
    return BeautifulSoup(resp.text, "lxml")

def absolute(url):
    if not url:
        return None
    return urljoin(BASE_URL + "/", url)

def clean(txt):
    if txt is None:
        return ""
    return re.sub(r"\s+", " ", txt).strip()

def parse_qs_from_url(href):
    try:
        return {k: v[0] for k, v in parse_qs(urlparse(href).query).items()}
    except Exception:
        return {}

print("Helpers HTTP pronti.")

## 4. Discovery — esplorazione struttura HTML degli endpoint

Prima di scrapare a tappeto, mappiamo form, parametri GET/POST e tabelle
presenti su ciascuna pagina.

In [ ]:
ENDPOINTS = {
    "horses":    "/db/db_horses.php",
    "results":   "/db/db_results.php",
    "champions": "/db/db_champions.php",
    "db":        "/db/db.php",
    "live":      "/live/live.php",
}

def explore_endpoint(name, path):
    url = BASE_URL + path
    print(f"\n--- {name.upper():10s} {url}")
    r = fetch(url)
    if r is None:
        print("  (irraggiungibile)")
        return
    s = soup_of(r)
    print(f"  status=200 size={len(r.text)} title={clean(s.title.text) if s.title else '?'}")
    forms = s.find_all("form")
    print(f"  forms: {len(forms)}")
    for f in forms[:3]:
        action = f.get("action") or path
        method = (f.get("method") or "GET").upper()
        inputs = []
        for inp in f.find_all(["input", "select", "textarea"]):
            inputs.append((inp.get("name"), inp.get("type", inp.name), inp.get("value", "")))
        print(f"    -> {method} {action}  fields={inputs[:8]}")
    tables = s.find_all("table")
    print(f"  tables: {len(tables)} (prima ha {len(tables[0].find_all('tr')) if tables else 0} righe)")
    # Link interni utili (con query string)
    qlinks = [a.get("href") for a in s.find_all("a", href=True) if "?" in a.get("href", "")]
    print(f"  link con query string: {len(qlinks)} (esempi: {qlinks[:3]})")

for name, path in ENDPOINTS.items():
    explore_endpoint(name, path)

In [ ]:
# Probe automatico di parametri GET comuni sul page horses.
# Stampiamo size della risposta per ciascuna variante: quelle che cambiano
# significativamente sono probabilmente i parametri "veri".
def probe_params(path, candidates):
    base = fetch(BASE_URL + path)
    base_size = len(base.text) if base is not None else 0
    print(f"baseline {path}: {base_size} bytes")
    for params in candidates:
        r = fetch(BASE_URL + path, params=params)
        sz = len(r.text) if r is not None else 0
        delta = sz - base_size
        flag = "*" if abs(delta) > 200 else " "
        print(f" {flag} params={params!s:55s} size={sz} delta={delta:+d}")

probe_params("/db/db_horses.php", [
    {"ricerca_n": "A"}, {"ricerca_n": "B"},
    {"page": 2}, {"pagina": 2},
    {"anno": 2023}, {"year": 2023},
])
probe_params("/db/db_results.php", [
    {"anno": 2023}, {"year": 2023}, {"a": 2023},
    {"page": 2}, {"pagina": 2},
])
probe_params("/db/db_champions.php", [
    {"v_filtro": "Open Riders World Ranking"},
    {"v_filtro": "Open Horses World Ranking"},
])

## 5. Parser tabelle generico

Il sito espone i dati in tabelle HTML. Una funzione generica copre la
maggior parte dei casi; estrattori specifici aggiungono i campi di dominio.

In [ ]:
def parse_html_tables(soup, min_rows=2, min_cols=2):
    """Ritorna una lista di tabelle (list[dict])."""
    out = []
    if soup is None:
        return out
    for tbl in soup.find_all("table"):
        rows = tbl.find_all("tr")
        if len(rows) < min_rows:
            continue
        # header
        head_cells = rows[0].find_all(["th", "td"])
        headers = [clean(c.get_text(" ")) for c in head_cells]
        if len(headers) < min_cols:
            continue
        # de-dup header names
        seen = {}
        norm_headers = []
        for h in headers:
            h = h or "col"
            seen[h] = seen.get(h, 0) + 1
            norm_headers.append(h if seen[h] == 1 else f"{h}_{seen[h]}")
        records = []
        for r in rows[1:]:
            cells_ = r.find_all(["td", "th"])
            if not cells_:
                continue
            rec = {}
            for i, c in enumerate(cells_[: len(norm_headers)]):
                key = norm_headers[i]
                rec[key] = clean(c.get_text(" "))
                # cattura anche eventuali link/parametri
                a = c.find("a", href=True)
                if a is not None:
                    rec[key + "__href"] = absolute(a["href"])
            if any(v for v in rec.values()):
                records.append(rec)
        if records:
            out.append({"headers": norm_headers, "rows": records})
    return out

print("parse_html_tables pronto.")

## 6. Scraper — Anagrafica cavalli (`db_horses.php`)

Iteriamo sulle lettere A–Z. Per ciascuna pagina raccogliamo tutte le righe
della tabella. Se un cavallo ha un link di dettaglio, lo seguiamo per
estrarre la genealogia (padre/madre, microchip, ecc.).

In [ ]:
HORSE_DETAIL_FIELDS = {
    # mappa label HTML -> nome colonna CSV
    "nome": "name", "name": "name",
    "id": "horse_id", "id fei": "fei_id",
    "anno di nascita": "birth_year", "anno nascita": "birth_year",
    "year of birth": "birth_year",
    "razza": "breed", "breed": "breed",
    "sesso": "sex", "sex": "sex",
    "microchip": "microchip", "passaporto": "passport",
    "passport": "passport",
    "padre": "sire", "sire": "sire",
    "madre": "dam", "dam": "dam",
    "allevatore": "breeder", "breeder": "breeder",
    "proprietario": "owner", "owner": "owner",
    "nazione": "country", "country": "country",
}

def scrape_horse_detail(url):
    """Estrae i dettagli di un cavallo dalla sua pagina."""
    r = fetch(url)
    s = soup_of(r)
    if s is None:
        return {}
    rec = {"detail_url": url}
    # Pattern 1: tabelle con label/value sulle righe
    for tbl in s.find_all("table"):
        for row in tbl.find_all("tr"):
            cells_ = row.find_all(["td", "th"])
            if len(cells_) == 2:
                label = clean(cells_[0].get_text(" ")).rstrip(":").lower()
                value = clean(cells_[1].get_text(" "))
                if label in HORSE_DETAIL_FIELDS and value:
                    rec[HORSE_DETAIL_FIELDS[label]] = value
    # Pattern 2: dt/dd
    for dl in s.find_all("dl"):
        terms = dl.find_all("dt")
        defs  = dl.find_all("dd")
        for t, d in zip(terms, defs):
            label = clean(t.get_text(" ")).rstrip(":").lower()
            if label in HORSE_DETAIL_FIELDS:
                rec[HORSE_DETAIL_FIELDS[label]] = clean(d.get_text(" "))
    return rec

def scrape_horses_registry():
    all_rows = []
    detail_count = 0
    for letter in tqdm(LETTERS_TO_SCRAPE, desc="Horses A-Z"):
        url = BASE_URL + ENDPOINTS["horses"]
        r = fetch(url, params={"ricerca_n": letter})
        s = soup_of(r)
        tables = parse_html_tables(s)
        if not tables:
            continue
        # Scegliamo la tabella più grande
        tbl = max(tables, key=lambda t: len(t["rows"]))
        for row in tbl["rows"]:
            row["letter"] = letter
            all_rows.append(row)
            # Detail follow-up
            detail_link = next((v for k, v in row.items()
                                if k.endswith("__href") and "horse" in (v or "").lower()), None)
            if detail_link and (MAX_HORSE_DETAILS is None or detail_count < MAX_HORSE_DETAILS):
                det = scrape_horse_detail(detail_link)
                row.update({f"detail_{k}": v for k, v in det.items()})
                detail_count += 1
    df = pd.DataFrame(all_rows)
    return df

horses_df = scrape_horses_registry()
print("Cavalli raccolti:", len(horses_df))
horses_df.head()

## 7. Scraper — Risultati storici gare (`db_results.php`)

Iteriamo per anno (e per pagina, se la lista è paginata). Per ogni gara
individuiamo la pagina di dettaglio e ne estraiamo i risultati.

In [ ]:
RESULT_LABEL_MAP = {
    "pos": "position", "posizione": "position", "rank": "position",
    "cavaliere": "rider", "rider": "rider", "atleta": "rider",
    "cavallo": "horse", "horse": "horse",
    "tempo": "total_time", "tempo totale": "total_time", "time": "total_time",
    "vel": "avg_speed", "velocita": "avg_speed", "velocità": "avg_speed",
    "speed": "avg_speed",
    "stato": "status", "status": "status", "esito": "status",
    "categoria": "category", "category": "category",
    "km": "km", "distanza": "km",
    "nazione": "country", "country": "country", "nation": "country",
    "club": "club",
    "fei": "fei_id",
    "penalita": "penalties", "penalità": "penalties",
    "penalty": "penalties",
}

def normalize_result_row(row):
    rec = {}
    loop_idx = 0
    for k, v in row.items():
        if k.endswith("__href"):
            rec[k] = v
            continue
        kl = k.lower().strip().rstrip(":")
        if kl in RESULT_LABEL_MAP:
            rec[RESULT_LABEL_MAP[kl]] = v
        elif re.match(r"^(loop|fase|gate)\s*\d+", kl):
            loop_idx += 1
            rec[f"loop_{loop_idx}_time"] = v
        elif "recupero" in kl or "recovery" in kl or "hr" in kl:
            rec.setdefault("recovery_hr", v)
        else:
            rec[kl] = v
    return rec

def find_race_links(soup):
    """Estrae i link a pagine di dettaglio gara dalla pagina anno."""
    links = []
    if soup is None:
        return links
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "results" in href.lower() and ("id" in href.lower() or "gara" in href.lower()
                                          or "race" in href.lower() or "?" in href):
            links.append({
                "race_name": clean(a.get_text(" ")),
                "race_url": absolute(href),
                "params": parse_qs_from_url(href),
            })
    # de-dup per URL
    seen = set(); uniq = []
    for L in links:
        if L["race_url"] not in seen:
            seen.add(L["race_url"]); uniq.append(L)
    return uniq

def parse_race_meta(soup):
    """Best-effort: estrae nome gara, data, luogo, km, categoria, nazione dall'header."""
    meta = {}
    if soup is None:
        return meta
    text = clean(soup.get_text(" ")[:1500])
    m = re.search(r"\b(\d{1,2}[./-]\d{1,2}[./-]\d{2,4})\b", text)
    if m: meta["race_date"] = m.group(1)
    m = re.search(r"\bCEI\s*\*{1,4}\b", text, re.I)
    if m: meta["category"] = m.group(0).upper().replace(" ", "")
    m = re.search(r"(\d{2,3})\s*km\b", text, re.I)
    if m: meta["km"] = m.group(1)
    if soup.title:
        meta["page_title"] = clean(soup.title.text)
    return meta

def scrape_results_for_year(year):
    """Ritorna (results_rows, races_rows) per un anno."""
    results_rows = []
    races_rows = []
    url = BASE_URL + ENDPOINTS["results"]
    page = 1
    seen_pages = set()
    while page <= MAX_PAGES_PER_LIST:
        r = fetch(url, params={"anno": year, "page": page})
        s = soup_of(r)
        if s is None:
            break
        # Detect end-of-pagination by content hashing
        sig = hash(s.get_text(" ")[:5000])
        if sig in seen_pages:
            break
        seen_pages.add(sig)

        race_links = find_race_links(s)
        if not race_links and page == 1:
            # Forse la pagina anno mostra direttamente i risultati in tabella
            for tbl in parse_html_tables(s):
                for row in tbl["rows"]:
                    rec = normalize_result_row(row)
                    rec["year"] = year
                    results_rows.append(rec)

        for race in race_links:
            race_meta = {"year": year, **race}
            r2 = fetch(race["race_url"])
            s2 = soup_of(r2)
            race_meta.update(parse_race_meta(s2))
            races_rows.append(race_meta)
            # estrai risultati dalla pagina gara
            for tbl in parse_html_tables(s2):
                for row in tbl["rows"]:
                    rec = normalize_result_row(row)
                    rec["year"] = year
                    rec["race_name"] = race.get("race_name")
                    rec["race_url"] = race["race_url"]
                    rec.update({f"race_{k}": v for k, v in race_meta.items()
                                if k in ("race_date", "category", "km", "page_title")})
                    results_rows.append(rec)

        if not race_links:
            break
        page += 1
    return results_rows, races_rows

all_results, all_races = [], []
for y in tqdm(YEARS_TO_SCRAPE, desc="Anni"):
    res, rcs = scrape_results_for_year(y)
    all_results.extend(res)
    all_races.extend(rcs)

results_df = pd.DataFrame(all_results)
races_df   = pd.DataFrame(all_races)
print("Righe risultati:", len(results_df), "| Gare:", len(races_df))
results_df.head()

## 8. Scraper — Rankings mondiali (`db_champions.php`)

In [ ]:
def scrape_rankings():
    rows = []
    url = BASE_URL + ENDPOINTS["champions"]
    for filt in tqdm(RANKING_FILTERS, desc="Rankings"):
        r = fetch(url, params={"v_filtro": filt})
        s = soup_of(r)
        tables = parse_html_tables(s)
        if not tables:
            continue
        tbl = max(tables, key=lambda t: len(t["rows"]))
        for row in tbl["rows"]:
            row["ranking_filter"] = filt
            row["scraped_at"] = dt.datetime.utcnow().isoformat(timespec="seconds")
            rows.append(row)
    return pd.DataFrame(rows)

rankings_df = scrape_rankings()
print("Righe ranking:", len(rankings_df))
rankings_df.head()

## 9. Live tracking (snapshot informativo)

Salviamo solo un'istantanea della pagina live (eventi attivi al momento
dell'esecuzione). Non viene esportato come CSV separato per default.

In [ ]:
def snapshot_live():
    r = fetch(BASE_URL + ENDPOINTS["live"])
    s = soup_of(r)
    if s is None:
        return []
    events = []
    for tbl in parse_html_tables(s):
        for row in tbl["rows"]:
            row["snapshot_at"] = dt.datetime.utcnow().isoformat(timespec="seconds")
            events.append(row)
    return events

live_events = snapshot_live()
print("Eventi live trovati:", len(live_events))

## 10. Esportazione CSV (UTF-8 BOM)

In [ ]:
def write_csv(df, name):
    if df is None or df.empty:
        print(f"  [skip] {name}: vuoto")
        return
    # Riordina mettendo le colonne 'utili' davanti
    preferred_first = ["year", "race_date", "race_name", "category", "km",
                       "position", "rider", "horse", "country", "club",
                       "total_time", "avg_speed", "status", "penalties",
                       "name", "horse_id", "fei_id", "birth_year", "breed",
                       "sex", "microchip", "passport", "sire", "dam",
                       "ranking_filter"]
    cols = [c for c in preferred_first if c in df.columns] + \
           [c for c in df.columns if c not in preferred_first]
    df = df[cols]
    path = OUT_DIR / name
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"  -> {path}  ({len(df)} righe, {len(df.columns)} colonne)")

print("Scrittura CSV in", OUT_DIR.resolve())
write_csv(horses_df,   "horses_registry.csv")
write_csv(results_df,  "results_full.csv")
write_csv(rankings_df, "rankings.csv")
write_csv(races_df,    "races_index.csv")

# Live snapshot opzionale
if live_events:
    write_csv(pd.DataFrame(live_events), "live_snapshot.csv")

## 11. Riepilogo finale

In [ ]:
summary = {
    "test_mode": TEST_MODE,
    "horses_rows":   len(horses_df),
    "results_rows":  len(results_df),
    "rankings_rows": len(rankings_df),
    "races_rows":    len(races_df),
    "years":         f"{YEARS_TO_SCRAPE[0]}-{YEARS_TO_SCRAPE[-1]}",
    "letters":       len(LETTERS_TO_SCRAPE),
    "output_dir":    str(OUT_DIR.resolve()),
    "completed_at":  dt.datetime.utcnow().isoformat(timespec="seconds"),
}
for k, v in summary.items():
    print(f"{k:>15}: {v}")